In [1]:
import pandas as pd
import numpy as np
import joblib

# =========================
# LOAD MODELS
# =========================
scaler = joblib.load("scaler.pkl")
kmedoids = joblib.load("kmedoids.pkl")

selected_features = [
    "loan_percent_income",
    "loan_intent_HOMEIMPROVEMENT",
    "loan_grade_B",
    "loan_grade_C",
    "loan_grade_E",
    "cb_person_default_on_file_Y"
]

# =========================
# USER INPUT
# =========================
user_data = {
    "person_age": float(input("Age: ")),
    "person_income": float(input("Income: ")),
    "person_emp_length": float(input("Employment length: ")),
    "loan_amnt": float(input("Loan amount: ")),
    "loan_int_rate": float(input("Interest rate: ")),
    "cb_person_cred_hist_length": float(input("Credit history length: ")),
    "loan_intent": input("Loan intent: ").upper(),
    "cb_person_default_on_file": input("Default (Y/N): ").upper()
}

df = pd.DataFrame([user_data])

# =========================
# FEATURE ENGINEERING
# =========================
df["loan_percent_income"] = df["loan_amnt"] / df["person_income"]

def assign_grade(ratio):
    if ratio <= 0.1:
        return "A"
    elif ratio <= 0.2:
        return "B"
    elif ratio <= 0.35:
        return "C"
    else:
        return "E"

df["loan_grade"] = df["loan_percent_income"].apply(assign_grade)

# =========================
# ENCODING
# =========================
df["loan_intent_HOMEIMPROVEMENT"] = (df["loan_intent"] == "HOMEIMPROVEMENT").astype(int)

df["loan_grade_B"] = (df["loan_grade"] == "B").astype(int)
df["loan_grade_C"] = (df["loan_grade"] == "C").astype(int)
df["loan_grade_E"] = (df["loan_grade"] == "E").astype(int)

df["cb_person_default_on_file_Y"] = (df["cb_person_default_on_file"] == "Y").astype(int)

# =========================
# NUMERIC SCALING
# =========================
numerical_cols = [
    'person_age',
    'person_income',
    'person_emp_length',
    'loan_amnt',
    'loan_int_rate',
    'loan_percent_income',
    'cb_person_cred_hist_length'
]

df_numeric = df[numerical_cols]

df_scaled_numeric = pd.DataFrame(
    scaler.transform(df_numeric),
    columns=numerical_cols
)

# =========================
# ATTACH FEATURES BACK
# =========================
df_scaled_numeric["loan_intent_HOMEIMPROVEMENT"] = df["loan_intent_HOMEIMPROVEMENT"]
df_scaled_numeric["loan_grade_B"] = df["loan_grade_B"]
df_scaled_numeric["loan_grade_C"] = df["loan_grade_C"]
df_scaled_numeric["loan_grade_E"] = df["loan_grade_E"]
df_scaled_numeric["cb_person_default_on_file_Y"] = df["cb_person_default_on_file_Y"]

# =========================
# FINAL INPUT (GA FEATURES)
# =========================
df_final = df_scaled_numeric[selected_features]

# =========================
# PREDICT CLUSTER
# =========================
cluster = kmedoids.predict(df_final)

print("\nFinal model input:")
print(df_final)

print("\nPredicted Cluster:", cluster[0])


Final model input:
   loan_percent_income  loan_intent_HOMEIMPROVEMENT  loan_grade_B  \
0             5.798993                            1             0   

   loan_grade_C  loan_grade_E  cb_person_default_on_file_Y  
0             0             1                            1  

Predicted Cluster: 1
